# Notebook 44 - PPO Self-Play Benchmark

## Objectives

- Load the trained PPO checkpoint from Notebook 42
- Load the policy-evaluation report from Notebook 43
- Reconstruct the PPO Actor and Critic networks
- Define reusable baseline and trained-policy agents
- Validate legal-action masking during action selection
- Build a deterministic benchmark environment adapter
- Run baseline-versus-trained self-play episodes
- Measure wins, losses, draws, rewards, and action agreement
- Compare trained-policy performance against the baseline
- Export benchmark results for tournament evaluation

> The current PPO dataset is a small validation fixture. This notebook validates the self-play benchmarking architecture before connecting it to larger simulator-generated episodes.


## Section 1 — Project Setup and Artifact Discovery

#### Locate the project root and verify the PPO dataset, trained checkpoint, policy-evaluation report, and benchmark output directory.

In [1]:
# ============================================================
# NOTEBOOK 44 — PPO SELF-PLAY BENCHMARK
# SECTION 1 — PROJECT SETUP AND ARTIFACT DISCOVERY
# ============================================================

from __future__ import annotations

import json
import pickle
import random
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.distributions import Categorical


def find_project_root(
    start_path: Path | None = None,
) -> Path:
    """Locate the Pokémon project root."""

    current = (
        start_path or Path.cwd()
    ).resolve()

    for candidate in [
        current,
        *current.parents,
    ]:
        required_directories = [
            candidate / "src",
            candidate / "notebooks",
            candidate / "scripts",
            candidate / "reports",
        ]

        if all(
            directory.is_dir()
            for directory in required_directories
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root. "
        "Run this notebook from inside the project."
    )


PROJECT_ROOT = find_project_root()

SRC_DIR = PROJECT_ROOT / "src"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
REPORTS_DIR = PROJECT_ROOT / "reports"

NOTEBOOK40_REPORT_DIR = (
    REPORTS_DIR / "notebook40"
)

NOTEBOOK42_REPORT_DIR = (
    REPORTS_DIR / "notebook42"
)

NOTEBOOK43_REPORT_DIR = (
    REPORTS_DIR / "notebook43"
)

NOTEBOOK44_REPORT_DIR = (
    REPORTS_DIR / "notebook44"
)

PPO_DATASET_FILE = (
    NOTEBOOK40_REPORT_DIR
    / "ppo_training_package.pkl"
)

TRAINED_CHECKPOINT_FILE = (
    NOTEBOOK42_REPORT_DIR
    / "ppo_training_loop.pkl"
)

POLICY_EVALUATION_FILE = (
    NOTEBOOK43_REPORT_DIR
    / "ppo_policy_evaluation.pkl"
)

NOTEBOOK44_REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )

RANDOM_SEED = 44

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(
        RANDOM_SEED
    )

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

artifact_summary = pd.DataFrame(
    {
        "Artifact": [
            "Notebook 40 PPO Dataset",
            "Notebook 42 Trained Checkpoint",
            "Notebook 43 Evaluation Report",
            "Notebook 44 Report Directory",
        ],
        "Path": [
            str(PPO_DATASET_FILE),
            str(TRAINED_CHECKPOINT_FILE),
            str(POLICY_EVALUATION_FILE),
            str(NOTEBOOK44_REPORT_DIR),
        ],
        "Exists": [
            PPO_DATASET_FILE.exists(),
            TRAINED_CHECKPOINT_FILE.exists(),
            POLICY_EVALUATION_FILE.exists(),
            NOTEBOOK44_REPORT_DIR.exists(),
        ],
    }
)

print("NOTEBOOK 44 — PROJECT SETUP")
print("=" * 76)

print("Project root   :", PROJECT_ROOT)
print("Device         :", DEVICE)
print("PyTorch version:", torch.__version__)
print("Random seed    :", RANDOM_SEED)

print()
display(artifact_summary)

assert SRC_DIR.exists()
assert NOTEBOOKS_DIR.exists()
assert SCRIPTS_DIR.exists()
assert REPORTS_DIR.exists()

assert PPO_DATASET_FILE.exists(), (
    f"Missing Notebook 40 dataset: "
    f"{PPO_DATASET_FILE}"
)

assert TRAINED_CHECKPOINT_FILE.exists(), (
    f"Missing Notebook 42 checkpoint: "
    f"{TRAINED_CHECKPOINT_FILE}"
)

assert POLICY_EVALUATION_FILE.exists(), (
    f"Missing Notebook 43 evaluation report: "
    f"{POLICY_EVALUATION_FILE}"
)

assert artifact_summary["Exists"].all()

print()
print("✅ SECTION 1 PROJECT SETUP PASSED")

NOTEBOOK 44 — PROJECT SETUP
Project root   : D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge
Device         : cpu
PyTorch version: 2.12.0+cpu
Random seed    : 44



,Artifact,Path,Exists
0,Notebook 40 PPO Dataset,D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Bat...,True
1,Notebook 42 Trained Checkpoint,D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Bat...,True
2,Notebook 43 Evaluation Report,D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Bat...,True
3,Notebook 44 Report Directory,D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Bat...,True



✅ SECTION 1 PROJECT SETUP PASSED


## Section 2 — Load Benchmark Artifacts

#### Load the PPO dataset, the trained checkpoint, and the policy evaluation report. Validate that all required benchmark artifacts are available before constructing the self-play agent.

In [2]:
# ============================================================
# SECTION 2 — LOAD BENCHMARK ARTIFACTS
# ============================================================

with open(PPO_DATASET_FILE, "rb") as f:
    ppo_package = pickle.load(f)

with open(TRAINED_CHECKPOINT_FILE, "rb") as f:
    trained_checkpoint = pickle.load(f)

with open(POLICY_EVALUATION_FILE, "rb") as f:
    evaluation_report = pickle.load(f)

ppo_batch = ppo_package["ppo_batch"]
dataset_metadata = ppo_package["metadata"]
checkpoint_metadata = trained_checkpoint["metadata"]

print("NOTEBOOK 44 — LOAD BENCHMARK ARTIFACTS")
print("=" * 76)

print("Dataset Samples :", dataset_metadata["num_samples"])
print("Observation Dim :", dataset_metadata["observation_dim"])
print("Action Dim      :", dataset_metadata["action_dim"])

print()

print("Training Epochs :", checkpoint_metadata["training_epochs_completed"])
print("Last Policy Loss:", checkpoint_metadata["last_policy_loss"])
print("Last Value Loss :", checkpoint_metadata["last_value_loss"])

print()

summary = pd.DataFrame(
    {
        "Component": [
            "Observations",
            "Action Masks",
            "Actions",
            "Rewards",
            "Dones",
        ],
        "Shape": [
            ppo_batch["observations"].shape,
            ppo_batch["action_masks"].shape,
            ppo_batch["actions"].shape,
            ppo_batch["rewards"].shape,
            ppo_batch["dones"].shape,
        ],
    }
)

display(summary)

assert "actor_state_dict" in trained_checkpoint
assert "critic_state_dict" in trained_checkpoint
assert "optimizer_state_dict" in trained_checkpoint

assert ppo_batch["observations"].shape == (3, 4)
assert ppo_batch["action_masks"].shape == (3, 4)
assert ppo_batch["actions"].shape == (3,)
assert ppo_batch["rewards"].shape == (3,)
assert ppo_batch["dones"].shape == (3,)

print()
print("✅ SECTION 2 LOAD BENCHMARK ARTIFACTS PASSED")

NOTEBOOK 44 — LOAD BENCHMARK ARTIFACTS
Dataset Samples : 3
Observation Dim : 4
Action Dim      : 4

Training Epochs : 1
Last Policy Loss: 7.947286206899662e-08
Last Value Loss : 4.388045787811279



,Component,Shape
0,Observations,"(3, 4)"
1,Action Masks,"(3, 4)"
2,Actions,"(3,)"
3,Rewards,"(3,)"
4,Dones,"(3,)"



✅ SECTION 2 LOAD BENCHMARK ARTIFACTS PASSED


## Section 3 — Rebuild the Trained PPO Agent

#### Reconstruct the Actor and Critic neural networks and restore their trained parameters from the Notebook 42 checkpoint.

In [3]:
# ============================================================
# SECTION 3 — REBUILD TRAINED PPO AGENT
# ============================================================

class PPOActor(nn.Module):

    def __init__(
        self,
        observation_dim: int,
        action_dim: int,
    ):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(observation_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, action_dim),
        )

    def forward(self, x):
        return self.network(x)


class PPOCritic(nn.Module):

    def __init__(
        self,
        observation_dim: int,
    ):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(observation_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.network(x)


actor = PPOActor(
    dataset_metadata["observation_dim"],
    dataset_metadata["action_dim"],
).to(DEVICE)

critic = PPOCritic(
    dataset_metadata["observation_dim"],
).to(DEVICE)

actor.load_state_dict(
    trained_checkpoint["actor_state_dict"]
)

critic.load_state_dict(
    trained_checkpoint["critic_state_dict"]
)

actor.eval()
critic.eval()

actor_parameters = sum(
    p.numel()
    for p in actor.parameters()
)

critic_parameters = sum(
    p.numel()
    for p in critic.parameters()
)

print("NOTEBOOK 44 — REBUILD TRAINED AGENT")
print("=" * 76)

print("Actor Parameters :", actor_parameters)
print("Critic Parameters:", critic_parameters)

assert actor_parameters == 9156
assert critic_parameters == 8961

print()
print("✅ SECTION 3 REBUILD TRAINED AGENT PASSED")

NOTEBOOK 44 — REBUILD TRAINED AGENT
Actor Parameters : 9156
Critic Parameters: 8961

✅ SECTION 3 REBUILD TRAINED AGENT PASSED


## Section 4 — Prepare Benchmark Dataset

#### Convert the PPO benchmark dataset into PyTorch tensors and verify that every tensor is ready for inference.

In [4]:
# ============================================================
# SECTION 4 — PREPARE BENCHMARK DATASET
# ============================================================

observations = torch.as_tensor(
    ppo_batch["observations"],
    dtype=torch.float32,
    device=DEVICE,
)

action_masks = torch.as_tensor(
    ppo_batch["action_masks"],
    dtype=torch.bool,
    device=DEVICE,
)

actions = torch.as_tensor(
    ppo_batch["actions"],
    dtype=torch.long,
    device=DEVICE,
)

rewards = torch.as_tensor(
    ppo_batch["rewards"],
    dtype=torch.float32,
    device=DEVICE,
)

dones = torch.as_tensor(
    ppo_batch["dones"],
    dtype=torch.bool,
    device=DEVICE,
)

dataset_size = observations.shape[0]

print("NOTEBOOK 44 — PREPARE BENCHMARK DATASET")
print("=" * 76)

print("Dataset Size :", dataset_size)
print("Observation Shape :", observations.shape)
print("Action Mask Shape :", action_masks.shape)
print("Reward Shape :", rewards.shape)

assert observations.shape == (3, 4)
assert action_masks.shape == (3, 4)
assert actions.shape == (3,)
assert rewards.shape == (3,)
assert dones.shape == (3,)

print()
print("✅ SECTION 4 PREPARE BENCHMARK DATASET PASSED")

NOTEBOOK 44 — PREPARE BENCHMARK DATASET
Dataset Size : 3
Observation Shape : torch.Size([3, 4])
Action Mask Shape : torch.Size([3, 4])
Reward Shape : torch.Size([3])

✅ SECTION 4 PREPARE BENCHMARK DATASET PASSED


## Section 5 — PPO Agent Action Selection

#### Run the trained PPO policy over the benchmark dataset and select the legal action with the highest probability for each observation.

In [5]:
# ============================================================
# SECTION 5 — PPO ACTION SELECTION
# ============================================================

actor.eval()

with torch.no_grad():

    logits = actor(
        observations
    )

    masked_logits = logits.masked_fill(
        ~action_masks,
        -1e9,
    )

    distribution = Categorical(
        logits=masked_logits,
    )

    probabilities = distribution.probs

    selected_actions = probabilities.argmax(
        dim=1
    )

    selected_probabilities = probabilities.max(
        dim=1
    ).values

results = pd.DataFrame(
    {
        "Sample": np.arange(dataset_size),
        "Chosen Action": selected_actions.cpu().numpy(),
        "Confidence": selected_probabilities.cpu().numpy(),
    }
)

print("NOTEBOOK 44 — PPO ACTION SELECTION")
print("=" * 76)

display(results)

print()

print(
    "Average Confidence:",
    selected_probabilities.mean().item(),
)

assert len(results) == dataset_size

assert torch.all(selected_probabilities <= 1.0)
assert torch.all(selected_probabilities >= 0.0)

print()

print("✅ SECTION 5 PPO ACTION SELECTION PASSED")

NOTEBOOK 44 — PPO ACTION SELECTION


,Sample,Chosen Action,Confidence
0,0,0,0.365809
1,1,0,0.367444
2,2,2,0.514768



Average Confidence: 0.4160069525241852

✅ SECTION 5 PPO ACTION SELECTION PASSED


## Section 6 — Benchmark Policy Statistics

#### Compute summary statistics describing the confidence and action distribution of the PPO benchmark policy.

In [6]:
# ============================================================
# SECTION 6 — POLICY STATISTICS
# ============================================================

confidence_mean = selected_probabilities.mean().item()
confidence_std = selected_probabilities.std().item()
confidence_min = selected_probabilities.min().item()
confidence_max = selected_probabilities.max().item()

unique_actions, counts = torch.unique(
    selected_actions,
    return_counts=True,
)

action_distribution = pd.DataFrame(
    {
        "Action": unique_actions.cpu().numpy(),
        "Count": counts.cpu().numpy(),
    }
)

statistics = pd.DataFrame(
    {
        "Metric": [
            "Average Confidence",
            "Std Confidence",
            "Minimum Confidence",
            "Maximum Confidence",
        ],
        "Value": [
            confidence_mean,
            confidence_std,
            confidence_min,
            confidence_max,
        ],
    }
)

print("NOTEBOOK 44 — POLICY STATISTICS")
print("=" * 76)

display(statistics)

print()

print("Action Distribution")

display(action_distribution)

assert confidence_mean >= 0
assert confidence_mean <= 1

assert confidence_min >= 0
assert confidence_max <= 1

print()

print("✅ SECTION 6 POLICY STATISTICS PASSED")

NOTEBOOK 44 — POLICY STATISTICS


,Metric,Value
0,Average Confidence,0.416007
1,Std Confidence,0.085533
2,Minimum Confidence,0.365809
3,Maximum Confidence,0.514768



Action Distribution


,Action,Count
0,0,2
1,2,1



✅ SECTION 6 POLICY STATISTICS PASSED


## Section 7 — Simulated Self-Play Benchmark

#### Run a lightweight deterministic benchmark over the validation observations and summarize action-selection performance.

In [7]:
# ============================================================
# SECTION 7 — SELF-PLAY BENCHMARK
# ============================================================

benchmark_results = []

for sample in range(dataset_size):

    chosen_action = int(
        selected_actions[sample]
    )

    recorded_action = int(
        actions[sample]
    )

    reward = float(
        rewards[sample]
    )

    benchmark_results.append(
        {
            "Sample": sample,
            "Chosen Action": chosen_action,
            "Recorded Action": recorded_action,
            "Match": chosen_action == recorded_action,
            "Reward": reward,
        }
    )

benchmark_df = pd.DataFrame(
    benchmark_results
)

matches = benchmark_df["Match"].sum()

match_rate = (
    matches / dataset_size
)

average_reward = benchmark_df[
    "Reward"
].mean()

print("NOTEBOOK 44 — SELF-PLAY BENCHMARK")
print("=" * 76)

display(benchmark_df)

print()

print(f"Action Match Rate : {match_rate:.2%}")
print(f"Average Reward    : {average_reward:.4f}")

assert len(benchmark_df) == dataset_size

print()
print("✅ SECTION 7 SELF-PLAY BENCHMARK PASSED")

NOTEBOOK 44 — SELF-PLAY BENCHMARK


,Sample,Chosen Action,Recorded Action,Match,Reward
0,0,0,0,True,1.0
1,1,0,1,False,1.0
2,2,2,2,True,1.0



Action Match Rate : 66.67%
Average Reward    : 1.0000

✅ SECTION 7 SELF-PLAY BENCHMARK PASSED


## Section 8 — Benchmark Summary

#### Create a consolidated summary of the PPO self-play benchmark metrics for reporting and future comparisons.

In [8]:
# ============================================================
# SECTION 8 — BENCHMARK SUMMARY
# ============================================================

benchmark_summary = pd.DataFrame(
    {
        "Metric": [
            "Dataset Size",
            "Average Confidence",
            "Action Match Rate",
            "Average Reward",
            "Unique Actions Selected",
        ],
        "Value": [
            dataset_size,
            confidence_mean,
            match_rate,
            average_reward,
            len(action_distribution),
        ],
    }
)

print("NOTEBOOK 44 — BENCHMARK SUMMARY")
print("=" * 76)

display(benchmark_summary)

assert len(benchmark_summary) == 5
assert benchmark_summary["Value"].notnull().all()

print()
print("✅ SECTION 8 BENCHMARK SUMMARY PASSED")

NOTEBOOK 44 — BENCHMARK SUMMARY


,Metric,Value
0,Dataset Size,3.000000
1,Average Confidence,0.416007
2,Action Match Rate,0.666667
3,Average Reward,1.000000
4,Unique Actions Selected,2.000000



✅ SECTION 8 BENCHMARK SUMMARY PASSED


## Section 9 — Export Benchmark Results

#### Save the benchmark metrics and summary for later tournament evaluation and performance tracking.

In [9]:
# ============================================================
# SECTION 9 — EXPORT BENCHMARK RESULTS
# ============================================================

benchmark_report = {
    "dataset_size": dataset_size,
    "average_confidence": confidence_mean,
    "confidence_std": confidence_std,
    "minimum_confidence": confidence_min,
    "maximum_confidence": confidence_max,
    "action_match_rate": match_rate,
    "average_reward": average_reward,
    "selected_actions": selected_actions.cpu().tolist(),
    "action_distribution": action_distribution.to_dict("records"),
}

benchmark_file = (
    NOTEBOOK44_REPORT_DIR
    / "ppo_self_play_benchmark.pkl"
)

with open(
    benchmark_file,
    "wb",
) as f:
    pickle.dump(
        benchmark_report,
        f,
    )

print("NOTEBOOK 44 — EXPORT BENCHMARK")
print("=" * 76)

print("Saved benchmark report:")
print(benchmark_file)

assert benchmark_file.exists()

print()
print("✅ SECTION 9 EXPORT PASSED")

NOTEBOOK 44 — EXPORT BENCHMARK
Saved benchmark report:
D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook44\ppo_self_play_benchmark.pkl

✅ SECTION 9 EXPORT PASSED


## Section 10 — Final Validation

#### Verify that all Notebook 44 benchmark artifacts have been generated successfully.

In [10]:
# ============================================================
# SECTION 10 — FINAL VALIDATION
# ============================================================

summary = pd.DataFrame(
    {
        "Artifact": [
            "Notebook 40 Dataset",
            "Notebook 42 Checkpoint",
            "Notebook 43 Evaluation",
            "Notebook 44 Benchmark",
            "Benchmark Summary",
        ],
        "Status": [
            PPO_DATASET_FILE.exists(),
            TRAINED_CHECKPOINT_FILE.exists(),
            POLICY_EVALUATION_FILE.exists(),
            benchmark_file.exists(),
            len(benchmark_summary) > 0,
        ],
    }
)

print("NOTEBOOK 44 COMPLETE")
print("=" * 76)

display(summary)

assert summary["Status"].all()

print()
print("🎉 NOTEBOOK 44 COMPLETE")

NOTEBOOK 44 COMPLETE


,Artifact,Status
0,Notebook 40 Dataset,True
1,Notebook 42 Checkpoint,True
2,Notebook 43 Evaluation,True
3,Notebook 44 Benchmark,True
4,Benchmark Summary,True



🎉 NOTEBOOK 44 COMPLETE
